# channel-list-reverse-build — ex1: build symmetric encoder/decoder channel pairs from a reversed list

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `channel-list-reverse-build`. Running the final beacon cell reports progress against the `GAN: channel-list reverse build` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: channel-list reverse build` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`channel-list-reverse-build`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "channel-list-reverse-build"
DD_SUBTOPIC = "GAN: channel-list reverse build"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Channel-list reverse build (encoder → decoder) — quick refresher

A symmetric autoencoder / DCGAN-style architecture uses the encoder's channel list, REVERSED, for the decoder. The convention reads naturally — the decoder unrolls the encoder.

```python
encoder_channels = [3, 64, 128, 256, 512]       # input → bottleneck
decoder_channels = encoder_channels[::-1]       # [512, 256, 128, 64, 3]
```

Then you walk consecutive pairs to build blocks:
```python
for in_c, out_c in zip(decoder_channels[:-1], decoder_channels[1:]):
    blocks.append(convt_block(in_c, out_c))
```

**Slice `[::-1]` vs `list(reversed(x))`.** Both work for a Python list. The slice form is one token shorter and emphasizes the structural symmetry — encoder is `c`, decoder is `c[::-1]`. ARENA uses the slice form.

**Why this matters.** Hardcoding two parallel lists is a recipe for drift: change the encoder, forget the decoder, mysterious shape mismatch three commits later. Deriving one from the other guarantees they stay in sync.

### Exercise 1 — build symmetric encoder/decoder channel pairs from a reversed list

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply slice-reverse `channels[::-1]` and `zip(c[:-1], c[1:])` to derive symmetric (in_c, out_c) decoder pairs from an encoder channel list.
> Keywords: dcgan, channels, symmetry, list-reverse
> ```

**KCs targeted:** `channel-list-reverse-slice`, `consecutive-pair-zip`

Implement `ex1_encoder_decoder_pairs(encoder_channels)`. The channel-symmetry pattern at the heart of DCGAN and U-Net architectures:

1. `encoder_channels` is a Python list of ints — e.g. `[3, 64, 128, 256, 512]` — read as 'input has 3 channels, after first block 64, ..., bottleneck 512'.
2. Build `encoder_pairs` by walking consecutive elements: zip `encoder_channels[:-1]` with `encoder_channels[1:]`. Each element is `(in_c, out_c)`. Return as a list of tuples.
3. Build `decoder_channels = encoder_channels[::-1]` (slice reverse — must be the slice form, not `list(reversed(...))`, so the structural symmetry shows in the code).
4. Build `decoder_pairs` by zipping consecutive elements of `decoder_channels`.
5. Return a dict `{'encoder_pairs': [...], 'decoder_pairs': [...]}`.

Input: `encoder_channels` — list of int, length >= 2.
Output: dict with two keys, each holding a list of `(in_c, out_c)` tuples.

The visualization renders the encoder/decoder channel pyramid as two stacked bar charts — width-decreasing for the encoder, width-increasing for the decoder — to show the symmetry.

In [ ]:
def ex1_encoder_decoder_pairs(encoder_channels: list[int]) -> dict:
    """Build (in, out) pair lists for encoder and reversed decoder."""
    raise NotImplementedError()


def _test_ex1():
    # Standard DCGAN-ish channel list.
    enc = [3, 64, 128, 256, 512]
    out = ex1_encoder_decoder_pairs(enc)
    assert isinstance(out, dict), f'expected dict, got {type(out).__name__}'
    assert set(out.keys()) == {'encoder_pairs', 'decoder_pairs'}, f'keys wrong: {set(out.keys())}'

    expected_enc = [(3, 64), (64, 128), (128, 256), (256, 512)]
    expected_dec = [(512, 256), (256, 128), (128, 64), (64, 3)]
    assert out['encoder_pairs'] == expected_enc, f'encoder_pairs wrong: {out["encoder_pairs"]}'
    assert out['decoder_pairs'] == expected_dec, f'decoder_pairs wrong: {out["decoder_pairs"]}'

    # Symmetry property — decoder pair i is encoder pair (-1-i) reversed.
    for i, (in_c, out_c) in enumerate(out['decoder_pairs']):
        sym_enc = out['encoder_pairs'][-1 - i]
        assert (in_c, out_c) == (sym_enc[1], sym_enc[0]), (
            f'symmetry broken at decoder pair {i}: {(in_c, out_c)} vs {sym_enc}'
        )

    # Different list shapes.
    short = [1, 4]
    s = ex1_encoder_decoder_pairs(short)
    assert s['encoder_pairs'] == [(1, 4)] and s['decoder_pairs'] == [(4, 1)]

    long = [1, 2, 4, 8, 16, 32]
    l = ex1_encoder_decoder_pairs(long)
    assert l['encoder_pairs'] == [(1, 2), (2, 4), (4, 8), (8, 16), (16, 32)]
    assert l['decoder_pairs'] == [(32, 16), (16, 8), (8, 4), (4, 2), (2, 1)]

    # Input list MUST NOT be mutated.
    snapshot = enc.copy()
    ex1_encoder_decoder_pairs(enc)
    assert enc == snapshot, 'input list must not be mutated'

    # --- Visualization: encoder pyramid (shrinking spatial) + decoder pyramid (growing spatial) ---
    viz_enc = [3, 64, 128, 256, 512, 1024]
    viz_out = ex1_encoder_decoder_pairs(viz_enc)
    enc_pairs = viz_out['encoder_pairs']; dec_pairs = viz_out['decoder_pairs']
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
    ax1.bar(range(len(enc_pairs)), [p[1] for p in enc_pairs], color='steelblue', edgecolor='black')
    for i, (ic, oc) in enumerate(enc_pairs):
        ax1.text(i, oc + 20, f'{ic}->{oc}', ha='center', fontsize=9)
    ax1.set_title('encoder: channel growth')
    ax1.set_ylabel('out channels')
    ax2.bar(range(len(dec_pairs)), [p[1] for p in dec_pairs], color='coral', edgecolor='black')
    for i, (ic, oc) in enumerate(dec_pairs):
        ax2.text(i, oc + 20, f'{ic}->{oc}', ha='center', fontsize=9)
    ax2.set_title('decoder: channel shrink (encoder reversed)')
    ax2.set_xlabel('block index'); ax2.set_ylabel('out channels')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_encoder_decoder_pairs(encoder_channels: list[int]) -> dict:
    encoder_pairs = list(zip(encoder_channels[:-1], encoder_channels[1:]))
    decoder_channels = encoder_channels[::-1]
    decoder_pairs = list(zip(decoder_channels[:-1], decoder_channels[1:]))
    return {'encoder_pairs': encoder_pairs, 'decoder_pairs': decoder_pairs}
```

**`zip(c[:-1], c[1:])` is the consecutive-pair idiom.** It produces `[(c0, c1), (c1, c2), ..., (c_{n-2}, c_{n-1})]` — the edges of a path graph over the list. Use this for any 'walk consecutive elements' loop, not just channels.

**`[::-1]` produces a NEW list.** The slice doesn't mutate the original, so callers can keep their encoder list intact. (Compare to `c.reverse()`, which mutates in place — bad idea here.)

**Why this matters operationally.** In a real network, you'd iterate `decoder_pairs` to build `nn.Sequential(*[convt_block(ic, oc) for ic, oc in decoder_pairs])`. The pair list is the contract between 'I know the channel plan' and 'I build the layers'.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()